# 01 - PubMedQA Dataset Exploration

This notebook loads the PubMedQA labeled dataset and explores its structure,
distributions, and characteristics before downstream processing.

**Outputs:** Dataset statistics, label distributions, context length analysis, MeSH term frequencies.

In [2]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
from collections import Counter
from src.data_loader import load_pubmedqa, get_dataset_stats, build_mesh_lookup, get_meshes_with_frequency

## Load Dataset

In [3]:
df = load_pubmedqa()
print(f"Dataset shape: {df.shape}")
df.head()

README.md: 0.00B [00:00, ?B/s]

pqa_labeled/train-00000-of-00001.parquet:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset shape: (1000, 5)


,pubid,question,context,long_answer,final_decision
0,21645374,Do mitochondria play a role in remodelling lac...,{'contexts': ['Programmed cell death (PCD) is ...,Results depicted mitochondrial dynamics in viv...,yes
1,16418930,Landolt C and snellen e acuity: differences in...,{'contexts': ['Assessment of visual acuity dep...,"Using the charts described, there was only a s...",no
2,9488747,"Syncope during bathing in infants, a pediatric...",{'contexts': ['Apparent life-threatening event...,"""Aquagenic maladies"" could be a pediatric form...",yes
3,17208539,Are the long-term results of the transanal pul...,{'contexts': ['The transanal endorectal pull-t...,Our long-term study showed significantly bette...,no
4,10808977,Can tailored interventions increase mammograph...,{'contexts': ['Telephone counseling and tailor...,The effects of the intervention were most pron...,yes


## Dataset Statistics

In [4]:
stats = get_dataset_stats(df)

print(f"Total samples: {stats['total_samples']}")
print(f"Mean context length: {stats['context_length_mean']:.3f}")
print(f"\nLabel distribution: {stats['label_distribution']}")
print(f"\nContext length distribution: {stats['context_length_distribution']}")
print(f"\nUnique MeSH terms: {stats['unique_meshes']}")

Total samples: 1000
Mean context length: 3.358

Label distribution: {'yes': 552, 'no': 338, 'maybe': 110}

Context length distribution: {1: 1, 2: 69, 3: 720, 4: 101, 5: 34, 6: 47, 7: 22, 8: 5, 9: 1}

Unique MeSH terms: 3408


## Context Structure Inspection

In [5]:
# Inspect context structure for a single sample
sample = df.loc[0, "context"]
print("Keys:", list(sample.keys()))
print(f"\nNumber of contexts: {len(sample['contexts'])}")
print(f"Labels: {list(sample['labels'])}")
print(f"MeSH terms: {list(sample['meshes'])}")
print(f"\nFirst context (truncated): {sample['contexts'][0][:200]}...")

Keys: ['contexts', 'labels', 'meshes', 'reasoning_required_pred', 'reasoning_free_pred']

Number of contexts: 2
Labels: ['BACKGROUND', 'RESULTS']
MeSH terms: ['Alismataceae', 'Apoptosis', 'Cell Differentiation', 'Mitochondria', 'Plant Leaves']

First context (truncated): Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant co...


## Context Length Distribution

In [9]:
context_lengths = df["context"].apply(lambda x: len(x["contexts"]))
print(f"Context list number of items:\nMean: {context_lengths.mean():.2f}, Median: {context_lengths.median()}, Min: {context_lengths.min()}, Max: {context_lengths.max()}")
print(f"\nValue counts:\n{context_lengths.value_counts().sort_index()}")

Context list number of items:
Mean: 3.36, Median: 3.0, Min: 1, Max: 9

Value counts:
context
1      1
2     69
3    720
4    101
5     34
6     47
7     22
8      5
9      1
Name: count, dtype: int64


## Top MeSH Terms

In [10]:
print("Top 40 MeSH terms:")
for term, count in stats['top_meshes']:
    print(f"  {term}: {count}")

Top 40 MeSH terms:
  Humans: 959
  Female: 785
  Male: 703
  Middle Aged: 542
  Adult: 492
  Aged: 414
  Retrospective Studies: 237
  Adolescent: 204
  Aged, 80 and over: 172
  Treatment Outcome: 138
  Prospective Studies: 138
  Young Adult: 127
  Risk Factors: 121
  Child: 107
  Follow-Up Studies: 92
  Time Factors: 80
  Child, Preschool: 75
  Surveys and Questionnaires: 75
  Cross-Sectional Studies: 71
  Pregnancy: 65
  Sensitivity and Specificity: 64
  Cohort Studies: 62
  Prognosis: 61
  Predictive Value of Tests: 54
  Infant: 52
  Infant, Newborn: 49
  Risk Assessment: 48
  Prevalence: 48
  Postoperative Complications: 48
  Case-Control Studies: 48
  Reproducibility of Results: 47
  Severity of Illness Index: 46
  Animals: 44
  Age Factors: 43
  United States: 43
  Survival Rate: 43
  Neoplasm Staging: 42
  Breast Neoplasms: 36
  Tomography, X-Ray Computed: 33
  Logistic Models: 31


In [11]:
# Mid-frequency MeSH terms (potential specialized domains)
mid_freq = get_meshes_with_frequency(df, min_freq=5, max_freq=20)
print(f"MeSH terms with frequency 5-20: {len(mid_freq)} terms")
list(mid_freq.items())[:20]

MeSH terms with frequency 5-20: 333 terms


[('Registries', 20),
 ('Sex Factors', 20),
 ('Feasibility Studies', 19),
 ('Health Knowledge, Attitudes, Practice', 19),
 ('Analysis of Variance', 19),
 ('Pain Measurement', 19),
 ('Stroke', 19),
 ('Myocardial Infarction', 19),
 ("Practice Patterns, Physicians'", 18),
 ('Ultrasonography', 18),
 ('Comorbidity', 18),
 ('Immunohistochemistry', 18),
 ('Obesity', 18),
 ('Statistics, Nonparametric', 18),
 ('Biomarkers', 17),
 ('Disease-Free Survival', 17),
 ('Disease Progression', 17),
 ('Double-Blind Method', 17),
 ('Hypertension', 17),
 ('Proportional Hazards Models', 16)]

## Build MeSH Lookup

This mapping is used later for metadata enrichment during chunking.

In [12]:
mesh_lookup = build_mesh_lookup(df)
print(f"MeSH lookup entries: {len(mesh_lookup)}")
print(f"Sample: pubid={list(mesh_lookup.keys())[0]} -> {list(mesh_lookup.values())[0][:5]}...")

MeSH lookup entries: 1000
Sample: pubid=21645374 -> ['Alismataceae' 'Apoptosis' 'Cell Differentiation' 'Mitochondria'
 'Plant Leaves']...
